<a href="https://colab.research.google.com/github/pranav-358/amazon_ml_challenge/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gc
import re
import unicodedata
import pandas as pd
from pathlib import Path
from google.colab import drive

# 1. MOUNT & CREATE ISOLATED WORKSPACE
drive.mount('/content/drive')
SHARED_PATH = Path("/content/drive/MyDrive/Amazon ML")
TRAIN_PATH = SHARED_PATH / "Data" / "train"

# This guarantees your files never conflict
MY_WORKSPACE = SHARED_PATH / "Pranav_Workspace"
MY_PROCESSED = MY_WORKSPACE / "processed"
MY_OUTPUT = MY_WORKSPACE / "output"
MY_MODELS = MY_WORKSPACE / "models"

for folder in [MY_PROCESSED, MY_OUTPUT, MY_MODELS]:
    folder.mkdir(parents=True, exist_ok=True)
print(f"Isolated Workspace Ready at: {MY_WORKSPACE}")

# 2. CLEANING FUNCTIONS
LEGAL_MAP = {"pvt": "private", "ltd": "limited", "inc": "incorporated", "corp": "corporation", "co": "company", "llc": "limited liability company"}
ADDRESS_MAP = {"rd": "road", "st": "street", "ave": "avenue", "blvd": "boulevard", "dr": "drive", "ctr": "center", "apt": "apartment", "opp": "opposite"}

def clean_text(text, type_map):
    if pd.isna(text) or not str(text).strip(): return ""
    text = "".join([c for c in unicodedata.normalize('NFKD', str(text)) if not unicodedata.combining(c)]).lower()
    text = re.sub(r"&", " and ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join([type_map.get(w, w) for w in text.split()]).strip()

# 3. PROCESS & COMPRESS (Memory Safe)
def process_and_save_isolated(filename, is_ground_truth=False):
    in_path = TRAIN_PATH / filename
    out_path = MY_PROCESSED / filename.replace('.tsv', '_clean.parquet')

    print(f"Processing {filename}...")
    df = pd.read_csv(in_path, sep="\t", dtype=str)

    if not is_ground_truth:
        # Strip rogue text from IDs instantly
        df["entity_id"] = df["entity_id"].astype(str).str.split().str[0]
        df["clean_name"] = df["business_name"].apply(lambda x: clean_text(x, LEGAL_MAP))
        df["clean_address"] = df["business_address"].apply(lambda x: clean_text(x, ADDRESS_MAP))
        df["blocking_key"] = df["clean_name"].str.split().str[0].fillna("")

        # Standardize country
        if "country" in df.columns:
            df["country"] = df["country"].str.lower().replace({"usa": "us", "ind": "india"})

    df.to_parquet(out_path, index=False)
    del df
    gc.collect()

process_and_save_isolated("train_source1.tsv")
process_and_save_isolated("train_source2.tsv")
process_and_save_isolated("train_source3.tsv")
process_and_save_isolated("train_ground_truth.tsv", is_ground_truth=True)

print("\nSuccess! All cleaned training data saved securely to your isolated folder.")

Mounted at /content/drive
Isolated Workspace Ready at: /content/drive/MyDrive/Amazon ML/Pranav_Workspace
Processing train_source1.tsv...
Processing train_source2.tsv...
Processing train_source3.tsv...
Processing train_ground_truth.tsv...

Success! All cleaned training data saved securely to your isolated folder.
